In [17]:
import numpy as np

# Input X: mỗi hàng là embedding vector của 1 token
# shape: (seq_len=3, d_model=2)
# d_model = 2: mỗi token được biểu diễn bằng vector 2 chiều
X = np.array([
    [1.0, 0.0],  # "I"
    [0.0, 1.0],  # "love"
    [1.0, 1.0],  # "AI"
])
print(X.shape)  # (3, 2)

(3, 2)


# Mini Self-Attention từ đầu

Triển khai Scaled Dot-Product Attention theo công thức:

```
Attention(Q, K, V) = softmax(Q @ K.T / sqrt(d_k)) @ V
```

**Luồng xử lý:**
1. Input X → chiếu tuyến tính ra Q, K, V
2. Tính scores = Q @ K.T (độ tương đồng giữa các token)
3. Scale scores / sqrt(d_k) để tránh softmax bão hòa
4. Softmax → attention weights (phân phối xác suất)
5. Context vectors = attention_weights @ V (tổng hợp thông tin)

In [18]:
# Weight matrices để chiếu X sang không gian Q, K, V
# shape: (d_model=2, d_k=2) — trong thực tế d_k thường nhỏ hơn d_model
# Ví dụ BERT: d_model=768, d_k=64 (12 heads)
# Trong training, các ma trận này được học qua backprop

W_q = np.array([   # chiếu X → Query (token đang "hỏi" thông tin gì)
    [1.0, 0.0],
    [0.0, 1.0],
])

W_K = np.array([   # chiếu X → Key (token "quảng cáo" thông tin của mình)
    [1.0, 0.0],
    [0.0, 1.0]
])

W_V = np.array([   # chiếu X → Value (nội dung thực sự được truyền đi)
    [0.5, 0.0],    # scale dim_0 xuống 0.5x
    [0.0, 2.0]     # scale dim_1 lên 2x
])

In [19]:
# Linear projection: chiếu X sang 3 không gian khác nhau
# Tương tự hệ thống tìm kiếm: Q=từ khóa, K=tiêu đề trang, V=nội dung trang
# shape sau phép nhân: (seq_len=3, d_k=2)
Q = X @ W_q   # Query: "Tôi đang tìm thông tin gì?"
K = X @ W_K   # Key:   "Tôi có thể cung cấp thông tin gì?"
V = X @ W_V   # Value: "Thông tin thực tế của tôi là gì?"

print("V:\n", V)

V:
 [[0.5 0. ]
 [0.  2. ]
 [0.5 2. ]]


In [20]:
# Dot-product scores: đo độ tương đồng giữa mọi cặp token (Q_i · K_j)
# scores[i][j] = token i chú ý vào token j bao nhiêu (chưa normalize)
# shape: (seq_len=3, seq_len=3)
scores = Q @ K.T
print("Scores:\n", scores)
print("K.shape:", K.shape)  # (seq_len, d_k)

Scores:
 [[1. 0. 1.]
 [0. 1. 1.]
 [1. 1. 2.]]
K.shape: (3, 2)


In [21]:
# Scale scores để tránh softmax bão hòa khi d_k lớn
# Khi d_k lớn, dot product có variance = d_k → chia sqrt(d_k) để chuẩn hóa variance về 1
# Công thức: scores / sqrt(d_k)
scaled_scores = scores / np.sqrt(K.shape[1])  # K.shape[1] = d_k = 2
print("Scaled Scores:\n", scaled_scores)

Scaled Scores:
 [[0.70710678 0.         0.70710678]
 [0.         0.70710678 0.70710678]
 [0.70710678 0.70710678 1.41421356]]


In [22]:
# Softmax theo hàng (axis=1): mỗi token i có tổng attention weights = 1
# Biến scores thô thành phân phối xác suất: token i chú ý bao nhiêu % vào token j
def softmax(x):
    exp_x = np.exp(x)
    return exp_x / np.sum(exp_x, axis=1, keepdims=True)

In [23]:
# attention_weights[i][j] = token i chú ý vào token j bao nhiêu %
# Mỗi hàng tổng = 1 (phân phối xác suất)
attention_weights = softmax(scaled_scores)
print("Attention Weights:\n", attention_weights)

Attention Weights:
 [[0.40111209 0.19777581 0.40111209]
 [0.19777581 0.40111209 0.40111209]
 [0.24825508 0.24825508 0.50348984]]


In [24]:
# Context vectors: biểu diễn mới của mỗi token sau khi tích hợp thông tin từ các token khác
# context[i] = weighted sum của tất cả V, trọng số là attention_weights[i]
# shape: (seq_len=3, d_v=2) — mỗi token giờ "nhìn thấy" toàn bộ câu
context_vectors = attention_weights @ V
print("Context Vectors:\n", context_vectors)

Context Vectors:
 [[0.40111209 1.19777581]
 [0.29944395 1.60444837]
 [0.37587246 1.50348984]]


In [25]:
# Causal Mask (Masked Self-Attention) — dùng trong GPT/decoder
# Mỗi token chỉ được nhìn về quá khứ, không được nhìn tương lai
# Lý do: khi sinh text, model chưa có các token phía sau → mask để training nhất quán với inference
#
# mask[i][j] = 0   nếu token i được phép nhìn token j (j <= i)
# mask[i][j] = -1e9 nếu bị che (j > i) → sau softmax: e^(-1e9) ≈ 0
mask = np.array([
    [0,   -1e9, -1e9],  # "I"    chỉ thấy chính nó
    [0,      0, -1e9],  # "love" thấy "I" và chính nó
    [0,      0,    0],  # "AI"   thấy tất cả
])

masked_scores = scaled_scores + mask
new_scores = softmax(masked_scores)
print("Masked Scores:\n", masked_scores)
print("New Attention Weights:\n", new_scores)

Masked Scores:
 [[ 7.07106781e-01 -1.00000000e+09 -9.99999999e+08]
 [ 0.00000000e+00  7.07106781e-01 -9.99999999e+08]
 [ 7.07106781e-01  7.07106781e-01  1.41421356e+00]]
New Attention Weights:
 [[1.         0.         0.        ]
 [0.33023845 0.66976155 0.        ]
 [0.24825508 0.24825508 0.50348984]]


In [26]:
print(new_scores.sum(axis=1))

[1. 1. 1.]
